# Day 1 / Session 2 -- Prompt Engineering Exercises

These exercises let you practice the six prompt engineering patterns from the demos.

| Exercise | Pattern(s) | Difficulty |
|----------|-----------|------------|
| 1 | Role-based personas (Demo 3) | Easy (Core) |
| 2 | Few-shot learning (Demo 1) | Easy (Core) |
| 3 | Chain-of-Thought (Demo 2) | Easy (Core) |
| Optional 1 | Multi-turn + JSON mode (Demos 4, 5) | Intermediate |
| Optional 2 | LangChain templates + Pydantic (Demo 6) | Intermediate |
| Optional 3 | All patterns combined | Intermediate |

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn langchain langchain-openai langchain-core python-dotenv

## Environment Setup

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv() # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import json

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

## Quick Recap

In the demos you saw six prompt-engineering patterns -- few-shot examples,
chain-of-thought reasoning, role-based personas, JSON-mode structured output,
multi-turn conversation management, and LangChain prompt templates.
Now you will apply those patterns yourself.

---
## Exercise 1: Design Effective System Prompts (Easy -- Core)

**Scenario.** You are building an **Industry Research Agent** that consulting
teams will query for fast, structured market intelligence. Your
job is to complete the system prompt that controls how the agent responds.

A template is provided with blanks for you to fill in.

> ** Key Concept:** A good system prompt has 4 parts: (1) Role definition,
> (2) Approach/methodology, (3) Output format, and (4) Behavioral constraints.

**Reference:** Demo 3 for persona definition pattern.

In [ ]:
# Exercise 1: Design Effective System Prompts for an Industry Research Agent
#
# A template is provided below. Fill in the blanks (marked with ___).
# The function ask_research_agent() is already implemented for you.

# TODO: Fill in the blanks in the system prompt template below.
#  Hint: Think about what makes a research agent useful to a consulting team.

research_agent_prompt = """You are a McKinsey Industry Research Agent specializing in
analyzing market dynamics, competitive landscapes, and strategic opportunities.

## Your Approach:
1. Break down research questions using ___ frameworks   # TODO: fill in (e.g., "MECE")
2. Support claims with ___ and ___                      # TODO: fill in (e.g., "data" and "evidence")
3. Distinguish between facts, estimates, and assumptions

## Output Format:
Always structure your response with these sections:
- **Executive Summary** (2-3 sentences)
- **Key Findings** (3-5 bullet points with supporting data)
- **Strategic Implications** (what this means for the client)
- **Confidence Level** (High / Medium / Low with justification)
- **Data Gaps** (what additional research is needed)

## Behavioral Guidelines:
- Acknowledge uncertainty explicitly
- Flag assumptions clearly
- Use specific numbers and percentages where possible
"""

def ask_research_agent(question):
    """Send a question to the research agent and print the response."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": research_agent_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "800"))
    )
    result = response.choices[0].message.content
    print(result)
    return result

# Test -- uncomment when ready
# ask_research_agent("What are the key growth drivers and competitive dynamics in the global electric vehicle battery market?")

### Hints (Exercise 1)

<details>
<summary>Hint 1: Filling in the blanks</summary>

The first blank should be a structured analysis methodology:
```
1. Break down research questions using MECE frameworks
```
The second blank pair should specify evidence types:
```
2. Support claims with data and quantitative evidence
```
</details>

<details>
<summary>Hint 2: Want to make the prompt even better?</summary>

Add industry-specific context or additional frameworks like Porter's Five Forces,
value chain analysis, or TAM/SAM/SOM market sizing.

You can also add more behavioral guidelines like:
- "Prioritize recent data (last 2-3 years)"
- "Include competitive benchmarking where relevant"
</details>

### Expected Output (Exercise 1)

When you run the cell above with a well-crafted system prompt, the agent
should return a response structured like this:

```
**Executive Summary**
The global EV battery market is projected to grow at ~25% CAGR through 2030,
driven by regulatory mandates, declining cell costs, and OEM electrification
commitments. Competitive intensity is high, with CATL, BYD, and LG Energy
Solution controlling >60% of global capacity.

**Key Findings**
- Global EV battery demand expected to reach ~3,500 GWh by 2030
- Top 3 manufacturers hold >60% market share; consolidation accelerating
- LFP chemistry gaining share over NMC due to cost/safety advantages
- Localization mandates (US IRA, EU Battery Regulation) reshaping supply chains

**Strategic Implications**
- Clients with cathode/anode material exposure should evaluate regional
 manufacturing footprint against emerging trade barriers
- OEMs face make-vs-buy decisions on cell production

**Confidence Level**: Medium -- public market data is robust, but proprietary
capacity-expansion plans introduce uncertainty.

**Data Gaps**
- Actual contracted vs. announced capacity by region
- Client-specific cost position vs. benchmark
```

### Reference Solution (Exercise 1)

Below is one reference implementation. Try your own first!

In [ ]:
# Reference solution -- try your own first!

research_agent_prompt_solution = """You are a McKinsey Industry Research Agent specializing in analyzing market dynamics, competitive landscapes, and strategic opportunities for client engagements.

## Your Approach:
1. Break down the research question into MECE sub-questions
2. Analyze each dimension with data-driven reasoning
3. Synthesize findings into a structured, partner-ready response

## Output Format:
Always structure your response as:
- **Executive Summary**: 2-3 sentence overview suitable for a CEO briefing
- **Key Findings**: Bullet points of main discoveries with quantitative evidence where possible
- **Strategic Implications**: What this means for the client's competitive position
- **Confidence Level**: High/Medium/Low with justification
- **Data Gaps**: What additional data would strengthen the analysis

## Guidelines:
- If uncertain, explicitly state your confidence level and flag assumptions
- Distinguish between facts, estimates, and inferences
- Use McKinsey-standard language and frameworks (Porter's Five Forces, value chain analysis, etc.)
- Provide balanced perspectives -- always consider the 'so what?' for the client"""

def ask_research_agent_solution(question):
 response = client.chat.completions.create(
 model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 messages=[
 {"role": "system", "content": research_agent_prompt_solution},
 {"role": "user", "content": question}
 ],
 max_tokens=800
 )
 result = response.choices[0].message.content
 print(result)
 return result

ask_research_agent_solution("What are the key growth drivers and competitive dynamics in the global electric vehicle battery market?")

---
## Exercise 2: Few-Shot Client Feedback Classifier (Easy -- Core)

**Scenario.** Your team receives client engagement feedback after every project.
You need to automatically classify each piece of feedback into one of 5 categories
so the firm can track trends across engagements.

**Categories:**
- `STRATEGY` -- feedback about strategic frameworks, market analysis, recommendations
- `OPERATIONS` -- feedback about process improvements, efficiency, supply chain
- `DELIVERY` -- feedback about timelines, milestones, project management
- `COMMUNICATION` -- feedback about presentations, status updates, stakeholder management
- `VALUE` -- feedback about ROI, cost savings, measurable impact

Most of the code is provided. You just need to add a few more few-shot examples.

> ** Key Concept:** Few-shot learning = give the model examples of correct
> input->output pairs so it learns the pattern. More examples = better accuracy.

**Reference:** Demo 1 for the few-shot message format.

In [ ]:
# Exercise 2: Few-Shot Client Feedback Classifier
#
# 6 of the 10 few-shot examples are provided. Add 4 more (marked TODO).
# The classify function and test loop are already complete.

def classify_feedback(feedback_text):
    # System message (provided)
    system_msg = {
        "role": "system",
        "content": "You are a client feedback classifier. Classify each feedback into exactly one category: STRATEGY, OPERATIONS, DELIVERY, COMMUNICATION, or VALUE. Respond with ONLY the category name."
    }

    # Few-shot examples: each pair = (user message with feedback, assistant reply with label)
    few_shot_examples = [
        # [OK] STRATEGY examples (provided)
        {"role": "user", "content": "Feedback: 'The market entry framework identified three untapped segments we hadn't considered.'"},
        {"role": "assistant", "content": "STRATEGY"},
        {"role": "user", "content": "Feedback: 'The competitive analysis missed two key digital-native players.'"},
        {"role": "assistant", "content": "STRATEGY"},

        # [OK] OPERATIONS examples (provided)
        {"role": "user", "content": "Feedback: 'The supply chain redesign reduced lead times by 30%.'"},
        {"role": "assistant", "content": "OPERATIONS"},

        # TODO 1: Add ONE more OPERATIONS example (user + assistant message pair)
        #  Hint: mention something about process improvement, efficiency, or manufacturing


        # [OK] DELIVERY example (provided)
        {"role": "user", "content": "Feedback: 'The team hit every milestone on the project timeline.'"},
        {"role": "assistant", "content": "DELIVERY"},

        # TODO 2: Add ONE more DELIVERY example
        #  Hint: mention something about deadlines, deliverables, or project management


        # TODO 3: Add TWO COMMUNICATION examples (user + assistant pairs)
        #  Hint: mention presentations, status reports, or stakeholder updates


        # [OK] VALUE examples (provided)
        {"role": "user", "content": "Feedback: 'The recommendations generated $5M in savings within 6 months.'"},
        {"role": "assistant", "content": "VALUE"},
        {"role": "user", "content": "Feedback: 'ROI on the engagement was 10x based on first-year results.'"},
        {"role": "assistant", "content": "VALUE"},
    ]

    # Build full message list and classify (provided -- no changes needed)
    messages = [system_msg] + few_shot_examples + [
        {"role": "user", "content": f"Feedback: '{feedback_text}'"}
    ]

    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=messages
    )
    return response.choices[0].message.content.strip()

# Test with 5 unlabeled statements
test_feedback = [
    "The team's weekly status reports were clear and well-organized.",
    "The cost reduction roadmap saved us $12M in the first year.",
    "The market entry framework was brilliant but the competitive analysis missed two key players.",
    "Deliverables were consistently late by 1-2 weeks throughout the engagement.",
    "The process redesign recommendations cut our cycle time by 40%."
]

# Uncomment when ready:
# for fb in test_feedback:
#     category = classify_feedback(fb)
#     print(f"{category:15s} | {fb}")

### Hints (Exercise 2)

<details>
<summary>Hint 1: Format of a few-shot example pair</summary>

Each example needs TWO messages -- a user message with the feedback and an assistant message with the label:
```python
{"role": "user", "content": "Feedback: 'The weekly presentations were engaging and clear.'"},
{"role": "assistant", "content": "COMMUNICATION"},
```
</details>

<details>
<summary>Hint 2: Example OPERATIONS feedback</summary>

```python
{"role": "user", "content": "Feedback: 'The manufacturing process optimization cut defect rates in half.'"},
{"role": "assistant", "content": "OPERATIONS"},
```
</details>

<details>
<summary>Hint 3: Example COMMUNICATION feedback</summary>

```python
{"role": "user", "content": "Feedback: 'The steering committee updates were concise and well-received.'"},
{"role": "assistant", "content": "COMMUNICATION"},
```
</details>

### Expected Output (Exercise 2)

```
COMMUNICATION | The team's weekly status reports were clear and well-organized.
VALUE | The cost reduction roadmap saved us $12M in the first year.
STRATEGY | The market entry framework was brilliant but the competitive analysis missed two key players.
DELIVERY | Deliverables were consistently late by 1-2 weeks throughout the engagement.
OPERATIONS | The process redesign recommendations cut our cycle time by 40%.
```

### Reference Solution (Exercise 2)

In [ ]:
# Reference solution -- try your own first!

def classify_feedback_solution(feedback_text):
 system_msg = {
 "role": "system",
 "content": "You are a client feedback classifier. Classify each feedback into exactly one category: STRATEGY, OPERATIONS, DELIVERY, COMMUNICATION, or VALUE. Respond with only the category name."
 }
 
 few_shot_examples = [
 # STRATEGY examples
 {"role": "user", "content": "Feedback: 'The strategic vision for digital transformation was exactly what our board needed.'"},
 {"role": "assistant", "content": "STRATEGY"},
 {"role": "user", "content": "Feedback: 'Their competitive landscape analysis uncovered three threats we had completely missed.'"},
 {"role": "assistant", "content": "STRATEGY"},
 
 # OPERATIONS examples
 {"role": "user", "content": "Feedback: 'The supply chain optimization playbook reduced our inventory holding costs significantly.'"},
 {"role": "assistant", "content": "OPERATIONS"},
 {"role": "user", "content": "Feedback: 'They helped us streamline the procurement workflow from 14 steps to 6.'"},
 {"role": "assistant", "content": "OPERATIONS"},
 
 # DELIVERY examples
 {"role": "user", "content": "Feedback: 'All three workstreams delivered their final reports on schedule.'"},
 {"role": "assistant", "content": "DELIVERY"},
 {"role": "user", "content": "Feedback: 'The interim deliverable was incomplete and had to be reworked.'"},
 {"role": "assistant", "content": "DELIVERY"},
 
 # COMMUNICATION examples
 {"role": "user", "content": "Feedback: 'The partner gave excellent executive presentations that kept our C-suite engaged.'"},
 {"role": "assistant", "content": "COMMUNICATION"},
 {"role": "user", "content": "Feedback: 'We appreciated the daily stand-ups and transparent progress updates.'"},
 {"role": "assistant", "content": "COMMUNICATION"},
 
 # VALUE examples
 {"role": "user", "content": "Feedback: 'The engagement generated a 4x return on our consulting investment within 18 months.'"},
 {"role": "assistant", "content": "VALUE"},
 {"role": "user", "content": "Feedback: 'Hard to justify the fee given the limited measurable business impact so far.'"},
 {"role": "assistant", "content": "VALUE"},
 ]
 
 messages = [system_msg] + few_shot_examples + [
 {"role": "user", "content": f"Feedback: '{feedback_text}'"}
 ]
 
 response = client.chat.completions.create(
 model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 messages=messages
 )
 return response.choices[0].message.content.strip()

# Test
test_feedback = [
 "The team's weekly status reports were clear and well-organized.",
 "The cost reduction roadmap saved us $12M in the first year.",
 "The market entry framework was brilliant but the competitive analysis missed two key players.",
 "Deliverables were consistently late by 1-2 weeks throughout the engagement.",
 "The process redesign recommendations cut our cycle time by 40%."
]

for fb in test_feedback:
 category = classify_feedback_solution(fb)
 print(f"{category:15s} | {fb}")

---
## Exercise 3: Chain-of-Thought Market Sizing (Easy -- Core)

**Scenario.** Market sizing is a core consulting skill. You need to build a
function that uses Chain-of-Thought prompting to walk through a market sizing
problem step by step.

The system prompt and CoT instructions are provided. You just need to
fill in the API call.

> ** Key Concept:** Chain-of-Thought (CoT) = tell the model to reason
> step-by-step before giving a final answer. This dramatically improves
> accuracy on analytical tasks.

**Reference:** Demo 2 for the CoT prompting pattern.

In [ ]:
# Exercise 3: Chain-of-Thought Market Sizing
#
# The system prompt and CoT instructions are provided.
# You only need to fill in the API call (TODO).

def market_size_estimator(question):
    # System prompt (provided)
    system_prompt = """You are a McKinsey market sizing specialist. Use structured,
data-driven reasoning to estimate market sizes. Always:
- State your assumptions explicitly
- Show the math at each step
- Distinguish between TAM (total), SAM (serviceable), and SOM (obtainable)
- Provide low / base / high range estimates"""

    # Chain-of-Thought instructions (provided)
    cot_instructions = """Let's size this market step by step:
1. Define the market scope and boundaries
2. Identify the key driver variables (population, adoption rate, price, etc.)
3. Estimate each variable with clear assumptions
4. Calculate the total addressable market (TAM)
5. Narrow to serviceable market (SAM) and obtainable market (SOM)
6. Provide low / base / high range estimates with confidence levels"""

    # Combine question with CoT instructions
    user_message = f"{question}\n\n{cot_instructions}"

    # TODO: Call the API and return the response text
    #  Hint: Same pattern as Exercise 1's ask_research_agent function
    #
    # response = client.chat.completions.create(
    #     model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    #     messages=[
    #         {"role": "system", "content": system_prompt},
    #         {"role": "user", "content": user_message}
    #     ],
    #     max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "1000"))
    # )
    # result = response.choices[0].message.content
    # print(result)
    # return result
    pass  # <-- remove this and uncomment the code above

# Test -- uncomment when ready:
# result = market_size_estimator("Estimate the total addressable market for AI-powered legal document review in the US")

### Hints (Exercise 3)

<details>
<summary>Hint 1: The API call pattern</summary>

This is the same pattern you've used in Exercise 1 and Session 1:
```python
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ],
    max_tokens=1000
)
result = response.choices[0].message.content
```
Just uncomment the provided code and remove `pass`.
</details>

<details>
<summary>Hint 2: Why does CoT work?</summary>

By asking the model to "think step by step", it allocates more tokens (compute) to
reasoning before producing the answer. This reduces errors on math and logic tasks.
The CoT instructions act as a scaffold that guides the model's reasoning process.
</details>

### Expected Output (Exercise 3)

Your function should produce a structured response like:

```
## Market Sizing: AI-Powered Legal Document Review (US)

### Step 1: Market Scope
- Focus: AI tools that automate review of contracts, litigation docs, regulatory filings
- Buyers: Law firms, corporate legal departments, compliance teams

### Step 2: Key Driver Variables
- Number of US law firms: ~450,000 (ABA data)
- Number of large corporate legal departments: ~10,000 (Fortune 10K)
- Annual spend on document review: varies by segment

### Step 3: Estimation
- Large law firms (AmLaw 200): 200 firms x $2M avg AI spend = $400M
- Mid-size firms (2,000 firms): 2,000 x $200K = $400M
- Corporate legal (10,000 depts): 10,000 x $150K = $1.5B
- Small firms / solo (remaining): limited adoption = $200M

### Step 4: Total Addressable Market
- TAM = $400M + $400M + $1.5B + $200M = ~$2.5B

### Step 5: Range Estimates
- Low: $1.5B (slower adoption, lower price points)
- Base: $2.5B (moderate adoption by 2028)
- High: $4.0B (accelerated adoption + expansion into adjacent tasks)

Assumptions: US market only, SaaS pricing model, current adoption ~15-20%
```

### Reference Solution (Exercise 3)

In [ ]:
# Reference solution -- try your own first!

def market_size_estimator_solution(question):
 system_prompt = """You are a McKinsey market sizing specialist. Use structured,
data-driven reasoning to estimate market sizes. Always:
- State your assumptions explicitly
- Show the math at each step
- Distinguish between TAM, SAM, and SOM where relevant
- Provide a final range estimate (low / base / high scenario)
- Use credible reference points and sanity checks"""
 
 cot_instructions = """Let's size this market step by step:
1. Define the market scope and boundaries
2. Identify the key driver variables (e.g., number of firms, spend per firm, adoption rate)
3. Estimate each variable with explicit assumptions
4. Calculate the total addressable market (TAM)
5. Provide low / base / high range estimates with the assumption driving each scenario"""
 
 user_message = f"{question}\n\n{cot_instructions}"
 
 response = client.chat.completions.create(
 model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 messages=[
 {"role": "system", "content": system_prompt},
 {"role": "user", "content": user_message}
 ],
 max_tokens=1000
 )
 result = response.choices[0].message.content
 print(result)
 return result

market_size_estimator_solution("Estimate the total addressable market for AI-powered legal document review in the US")

---
## Optional Exercise 1 (Intermediate): Multi-Turn Research Agent

**Scenario.** Combine the ConversationManager from Demo 5 with the research
agent prompt from Exercise 1 and the JSON structured output from Demo 4.
Build a research agent that can handle follow-up questions about the same
industry and return structured JSON when requested.

> ** Hint:** The `send()` method follows the exact same pattern as
> `SimpleAgent.chat()` from Session 1, Exercise 1: append user message ->
> call API -> append assistant reply -> return.

**References:** Demo 4 (JSON mode), Demo 5 (conversation management)

In [ ]:
# Optional Exercise 1: Multi-Turn Research Agent
#
# Step 1: Define ResearchConversationManager class
# Extend the ConversationManager pattern from Demo 5.
# Add __init__ with a research-focused system prompt.
#
# Step 2: Implement send() method (same as Demo 5)
# Append user message, call API, append assistant message, return.
#
# Step 3: Implement send_json() method
# Same as send(), but add an instruction to the user message
# requesting JSON output, and use response_format={"type": "json_object"}
# Parse the JSON before returning.
#
# Step 4: Test with 3-turn conversation

class ResearchConversationManager:
 def __init__(self):
 self.model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
 self.messages = [
 {"role": "system", "content": ""} # YOUR CODE HERE -- research agent system prompt
 ]
 self.client = openai.OpenAI()
 
 def send(self, user_message):
 # YOUR CODE HERE -- follow Demo 5 pattern
 pass
 
 def send_json(self, user_message):
 # YOUR CODE HERE
 # Add instruction: "Return your response as a JSON object with keys:
 # executive_summary, key_findings (list), risks (list), confidence (string)"
 # Use response_format={"type": "json_object"}
 # Parse with json.loads() before returning
 pass
 
 def get_history_length(self):
 return len(self.messages)

# Test -- uncomment when ready:
# agent = ResearchConversationManager()
# print("Turn 1:")
# print(agent.send("Analyze the competitive landscape for cloud computing in 2025"))
# print("\n" + "-"*50)
# print("Turn 2:")
# print(agent.send("Drill deeper into the AI/ML services segment specifically"))
# print("\n" + "-"*50)
# print("Turn 3 (JSON):")
# result = agent.send_json("What are the top 3 risks for a mid-size company entering this space?")
# print(json.dumps(result, indent=2))
# print(f"\nHistory length: {agent.get_history_length()} messages")

### Reference Solution (Optional Exercise 1)

In [ ]:
# Reference solution -- try your own first!

class ResearchConversationManagerSolution:
 def __init__(self):
 self.model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
 self.messages = [
 {"role": "system", "content": """You are a McKinsey Industry Research Agent specializing in analyzing market dynamics, competitive landscapes, and strategic opportunities.

## Your Approach:
1. Break down research questions into MECE sub-questions
2. Analyze each dimension with data-driven reasoning
3. Build on prior context from the conversation

## Output Format (when text):
- Executive Summary (2-3 sentences)
- Key Findings (bullet points)
- Strategic Implications
- Confidence Level (High/Medium/Low)

## Guidelines:
- Flag assumptions explicitly
- Use frameworks (Porter's Five Forces, value chain analysis)
- Build on prior conversation context"""}
 ]
 self.client = openai.OpenAI()
 
 def send(self, user_message):
 self.messages.append({"role": "user", "content": user_message})
 response = self.client.chat.completions.create(
 model=self.model,
 messages=self.messages,
 max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
 )
 assistant_msg = response.choices[0].message.content
 self.messages.append({"role": "assistant", "content": assistant_msg})
 return assistant_msg
 
 def send_json(self, user_message):
 json_instruction = user_message + "\n\nReturn your response as a JSON object with keys: executive_summary (string), key_findings (list of strings), risks (list of strings), confidence (string: High/Medium/Low)."
 self.messages.append({"role": "user", "content": json_instruction})
 response = self.client.chat.completions.create(
 model=self.model,
 messages=self.messages,
 max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500")),
 response_format={"type": "json_object"}
 )
 assistant_msg = response.choices[0].message.content
 self.messages.append({"role": "assistant", "content": assistant_msg})
 return json.loads(assistant_msg)
 
 def get_history_length(self):
 return len(self.messages)

agent = ResearchConversationManagerSolution()
print("Turn 1:")
print(agent.send("Analyze the competitive landscape for cloud computing in 2025"))
print("\n" + "-"*50)
print("Turn 2:")
print(agent.send("Drill deeper into the AI/ML services segment specifically"))
print("\n" + "-"*50)
print("Turn 3 (JSON):")
result = agent.send_json("What are the top 3 risks for a mid-size company entering this space?")
print(json.dumps(result, indent=2))
print(f"\nHistory length: {agent.get_history_length()} messages")

---
## Optional Exercise 2 (Intermediate): Prompt Template Library

**Scenario.** Build a reusable library of LangChain prompt templates for
common consulting tasks. Each template should accept relevant parameters and
return structured output via Pydantic.

**Your task:**
Create three template functions:
1. `market_analysis(industry, region)` -> MarketAnalysisOutput
2. `competitive_assessment(company, competitors_list)` -> CompetitiveAssessmentOutput
3. `risk_evaluation(initiative, risk_factors)` -> RiskEvaluationOutput

> ** Hint:** The LangChain chain pattern is: `prompt | llm | parser`.
> Each step feeds into the next using the `|` pipe operator.

**Reference:** Demo 6 for LangChain prompt templates + Pydantic output parsing.

In [ ]:
# Optional Exercise 2: Prompt Template Library
#
# Step 1: Import LangChain components (already imported if you ran Demo 6)
# Step 2: Define Pydantic models for each output type
# Step 3: Create ChatPromptTemplates for each task
# Step 4: Build chains (prompt | llm | parser)
# Step 5: Wrap each chain in a function

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# --- Pydantic Models ---

class MarketAnalysisOutput(BaseModel):
 market_size: str = Field(description="Estimated market size with currency and year")
 growth_rate: str = Field(description="Projected CAGR or growth rate")
 key_drivers: list[str] = Field(description="Top 3-5 market drivers")
 key_risks: list[str] = Field(description="Top 3 market risks")
 outlook: str = Field(description="1-2 sentence market outlook")

# YOUR CODE HERE -- define CompetitiveAssessmentOutput
# Fields: company_position (str), strengths (list[str]), weaknesses (list[str]),
# competitive_threats (list[str]), recommendation (str)

# YOUR CODE HERE -- define RiskEvaluationOutput
# Fields: overall_risk_level (str), risk_assessments (list[str]),
# mitigation_strategies (list[str]), go_no_go (str), rationale (str)

# --- Template Functions ---

def market_analysis(industry, region):
 # YOUR CODE HERE
 # 1. Create parser with MarketAnalysisOutput
 # 2. Create ChatPromptTemplate
 # 3. Build chain: template | llm | parser
 # 4. Invoke and return result
 pass

def competitive_assessment(company, competitors_list):
 # YOUR CODE HERE
 pass

def risk_evaluation(initiative, risk_factors):
 # YOUR CODE HERE
 pass

# Test -- uncomment when ready:
# print("=== Market Analysis ===")
# result1 = market_analysis("electric vehicles", "Southeast Asia")
# print(json.dumps(result1, indent=2))

# print("\n=== Competitive Assessment ===")
# result2 = competitive_assessment("Spotify", ["Apple Music", "YouTube Music", "Amazon Music"])
# print(json.dumps(result2, indent=2))

# print("\n=== Risk Evaluation ===")
# result3 = risk_evaluation("Launching a direct-to-consumer channel", ["channel conflict", "logistics complexity", "brand dilution"])
# print(json.dumps(result3, indent=2))

### Reference Solution (Optional Exercise 2)

In [ ]:
# Reference solution -- try your own first!

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

llm_sol = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# --- Pydantic Models ---

class MarketAnalysisOutputSol(BaseModel):
 market_size: str = Field(description="Estimated market size with currency and year")
 growth_rate: str = Field(description="Projected CAGR or growth rate")
 key_drivers: list[str] = Field(description="Top 3-5 market drivers")
 key_risks: list[str] = Field(description="Top 3 market risks")
 outlook: str = Field(description="1-2 sentence market outlook")

class CompetitiveAssessmentOutputSol(BaseModel):
 company_position: str = Field(description="Current competitive position summary")
 strengths: list[str] = Field(description="Key competitive strengths")
 weaknesses: list[str] = Field(description="Key competitive weaknesses")
 competitive_threats: list[str] = Field(description="Main competitive threats")
 recommendation: str = Field(description="Strategic recommendation in 1-2 sentences")

class RiskEvaluationOutputSol(BaseModel):
 overall_risk_level: str = Field(description="Overall risk: High, Medium, or Low")
 risk_assessments: list[str] = Field(description="Assessment of each risk factor")
 mitigation_strategies: list[str] = Field(description="Recommended mitigation for each risk")
 go_no_go: str = Field(description="Recommendation: GO, CONDITIONAL GO, or NO GO")
 rationale: str = Field(description="1-2 sentence rationale for the recommendation")

# --- Template Functions ---

def market_analysis_solution(industry, region):
 parser = JsonOutputParser(pydantic_object=MarketAnalysisOutputSol)
 template = ChatPromptTemplate.from_messages([
 ("system", "You are a McKinsey market research analyst. Provide data-driven market analysis."),
 ("user", "Analyze the {industry} market in {region}.\n\n{format_instructions}")
 ])
 chain = template | llm_sol | parser
 return chain.invoke({"industry": industry, "region": region, "format_instructions": parser.get_format_instructions()})

def competitive_assessment_solution(company, competitors_list):
 parser = JsonOutputParser(pydantic_object=CompetitiveAssessmentOutputSol)
 competitors_str = ", ".join(competitors_list)
 template = ChatPromptTemplate.from_messages([
 ("system", "You are a McKinsey competitive strategy expert. Provide rigorous competitive analysis."),
 ("user", "Assess {company}'s competitive position against: {competitors}.\n\n{format_instructions}")
 ])
 chain = template | llm_sol | parser
 return chain.invoke({"company": company, "competitors": competitors_str, "format_instructions": parser.get_format_instructions()})

def risk_evaluation_solution(initiative, risk_factors):
 parser = JsonOutputParser(pydantic_object=RiskEvaluationOutputSol)
 risks_str = ", ".join(risk_factors)
 template = ChatPromptTemplate.from_messages([
 ("system", "You are a McKinsey risk & resilience expert. Evaluate risks with actionable mitigation strategies."),
 ("user", "Evaluate the risks of: {initiative}\nKey risk factors to assess: {risk_factors}\n\n{format_instructions}")
 ])
 chain = template | llm_sol | parser
 return chain.invoke({"initiative": initiative, "risk_factors": risks_str, "format_instructions": parser.get_format_instructions()})

# Test
print("=== Market Analysis ===")
result1 = market_analysis_solution("electric vehicles", "Southeast Asia")
print(json.dumps(result1, indent=2))

print("\n=== Competitive Assessment ===")
result2 = competitive_assessment_solution("Spotify", ["Apple Music", "YouTube Music", "Amazon Music"])
print(json.dumps(result2, indent=2))

print("\n=== Risk Evaluation ===")
result3 = risk_evaluation_solution("Launching a direct-to-consumer channel", ["channel conflict", "logistics complexity", "brand dilution"])
print(json.dumps(result3, indent=2))

---
## Optional Exercise 3 (Intermediate): Auto-Prompt Optimizer

> **Note:** This exercise combines concepts from all 6 demos.

**Scenario.** Build a system that automatically improves prompts. Given a basic
prompt and a set of test cases (input + expected output), the system:
1. Runs the prompt on all test cases
2. Evaluates results against expected outputs
3. Uses CoT reasoning to analyze failures and generate an improved prompt
4. Tests the improved prompt to verify it scores better

> ** Hint:** This is a meta-learning pipeline: the LLM critiques its own
> prompt and suggests improvements. The `optimize_prompt` function asks the LLM
> to analyze what went wrong and rewrite the prompt accordingly.

**References:** Demo 2 (CoT), Demo 4 (JSON mode), Demo 5 (pipeline chaining)

In [ ]:
# Optional Exercise 3: Auto-Prompt Optimizer
#
# Step 1: Define test cases (input + expected output pairs)
# Step 2: Implement run_prompt(prompt, test_input) to get LLM output
# Step 3: Implement evaluate_results(test_cases, results) to score accuracy
# Step 4: Implement optimize_prompt(original_prompt, test_cases, results)
# to generate an improved prompt using CoT analysis
# Step 5: Compare original vs optimized prompt

def run_prompt(system_prompt, user_input):
 """Run a single prompt and return the response."""
 # YOUR CODE HERE
 pass

def evaluate_results(test_cases, results):
 """Score how well the results match expected outputs.
 Return a dict with: score (float 0-1), details (list of per-case results).
 Use JSON mode (Demo 4) to get structured evaluation from the LLM."""
 # YOUR CODE HERE
 pass

def optimize_prompt(original_prompt, test_cases, evaluation):
 """Use CoT (Demo 2) to analyze failures and generate an improved prompt.
 The optimizer LLM should:
 1. Analyze what the original prompt got wrong
 2. Identify patterns in the failures
 3. Generate an improved prompt that addresses the issues
 Return the improved prompt string."""
 # YOUR CODE HERE
 pass

def auto_optimize(original_prompt, test_cases):
 """Full optimization pipeline."""
 # Step 1: Run original prompt
 print("=== Running Original Prompt ===")
 original_results = []
 for tc in test_cases:
 result = run_prompt(original_prompt, tc["input"])
 original_results.append(result)
 print(f" Input: {tc['input'][:50]}...")
 print(f" Expected: {tc['expected'][:50]}...")
 print(f" Got: {result[:50] if result else 'None'}...")
 
 # Step 2: Evaluate original
 print("\n=== Evaluating Original Prompt ===")
 original_eval = evaluate_results(test_cases, original_results)
 print(f" Score: {original_eval}")
 
 # Step 3: Optimize
 print("\n=== Generating Improved Prompt ===")
 improved_prompt = optimize_prompt(original_prompt, test_cases, original_eval)
 print(f" Improved prompt: {improved_prompt[:100]}...")
 
 # Step 4: Run improved prompt
 print("\n=== Running Improved Prompt ===")
 improved_results = []
 for tc in test_cases:
 result = run_prompt(improved_prompt, tc["input"])
 improved_results.append(result)
 print(f" Input: {tc['input'][:50]}...")
 print(f" Got: {result[:50] if result else 'None'}...")
 
 # Step 5: Evaluate improved
 print("\n=== Evaluating Improved Prompt ===")
 improved_eval = evaluate_results(test_cases, improved_results)
 print(f" Score: {improved_eval}")
 
 return {
 "original_prompt": original_prompt,
 "improved_prompt": improved_prompt,
 "original_score": original_eval,
 "improved_score": improved_eval
 }

# Test cases for a sentiment classifier (intentionally weak original prompt)
test_cases = [
 {"input": "The project was delivered on time and under budget.", "expected": "POSITIVE"},
 {"input": "We lost three key team members during the engagement.", "expected": "NEGATIVE"},
 {"input": "The analysis was thorough but the timeline was tight.", "expected": "NEUTRAL"},
 {"input": "Revenue increased 25% after implementing the recommendations.", "expected": "POSITIVE"},
 {"input": "The final presentation lacked the depth our board expected.", "expected": "NEGATIVE"},
]

original_prompt = "Classify the sentiment." # Intentionally weak

# Uncomment when ready:
# results = auto_optimize(original_prompt, test_cases)
# print("\n=== Final Comparison ===")
# print(json.dumps(results, indent=2))

### Reference Solution (Optional Exercise 3)

In [ ]:
# Reference solution -- try your own first!

def run_prompt_solution(system_prompt, user_input):
 """Run a single prompt and return the response."""
 response = client.chat.completions.create(
 model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 messages=[
 {"role": "system", "content": system_prompt},
 {"role": "user", "content": user_input}
 ],
 max_tokens=50
 )
 return response.choices[0].message.content.strip()

def evaluate_results_solution(test_cases, results):
 """Score how well results match expected outputs using LLM as judge."""
 evaluation_prompt = """You are an evaluation judge. Compare each result against the expected output.
Return a JSON object with:
- "score": float from 0.0 to 1.0 (fraction of correct matches)
- "details": list of objects, each with "input", "expected", "actual", "correct" (boolean), "reason" (string)

Test cases and results:"""
 
 cases_str = ""
 for i, (tc, result) in enumerate(zip(test_cases, results)):
 cases_str += f"\nCase {i+1}: Input='{tc['input']}' | Expected='{tc['expected']}' | Actual='{result}'"
 
 response = client.chat.completions.create(
 model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 messages=[
 {"role": "system", "content": evaluation_prompt},
 {"role": "user", "content": cases_str}
 ],
 response_format={"type": "json_object"},
 max_tokens=500
 )
 return json.loads(response.choices[0].message.content)

def optimize_prompt_solution(original_prompt, test_cases, evaluation):
 """Use CoT to analyze failures and generate an improved prompt."""
 optimizer_prompt = """You are a prompt engineering expert. Analyze why the original prompt
failed on some test cases and generate an improved version.

Think step by step:
1. What did the original prompt get wrong?
2. What patterns do you see in the failures?
3. What specific instructions would fix these issues?
4. Write the improved prompt.

Return ONLY the improved system prompt text, nothing else."""
 
 context = f"""Original prompt: \"{original_prompt}\"

Evaluation results:
{json.dumps(evaluation, indent=2)}

Generate an improved system prompt that addresses the failures."""
 
 response = client.chat.completions.create(
 model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
 messages=[
 {"role": "system", "content": optimizer_prompt},
 {"role": "user", "content": context}
 ],
 max_tokens=300
 )
 return response.choices[0].message.content.strip()

def auto_optimize_solution(original_prompt, test_cases):
 """Full optimization pipeline."""
 # Step 1: Run original prompt
 print("=== Running Original Prompt ===")
 print(f"Prompt: '{original_prompt}'")
 original_results = []
 for tc in test_cases:
 result = run_prompt_solution(original_prompt, tc["input"])
 original_results.append(result)
 match = "PASS" if tc["expected"].upper() in result.upper() else "FAIL"
 print(f" [{match}] Input: {tc['input'][:60]}")
 print(f" Expected: {tc['expected']} | Got: {result[:60]}")
 
 # Step 2: Evaluate original
 print("\n=== Evaluating Original Prompt ===")
 original_eval = evaluate_results_solution(test_cases, original_results)
 print(f" Score: {original_eval.get('score', 'N/A')}")
 
 # Step 3: Optimize
 print("\n=== Generating Improved Prompt ===")
 improved_prompt = optimize_prompt_solution(original_prompt, test_cases, original_eval)
 print(f" Improved prompt: '{improved_prompt[:120]}...'")
 
 # Step 4: Run improved prompt
 print("\n=== Running Improved Prompt ===")
 improved_results = []
 for tc in test_cases:
 result = run_prompt_solution(improved_prompt, tc["input"])
 improved_results.append(result)
 match = "PASS" if tc["expected"].upper() in result.upper() else "FAIL"
 print(f" [{match}] Input: {tc['input'][:60]}")
 print(f" Expected: {tc['expected']} | Got: {result[:60]}")
 
 # Step 5: Evaluate improved
 print("\n=== Evaluating Improved Prompt ===")
 improved_eval = evaluate_results_solution(test_cases, improved_results)
 print(f" Score: {improved_eval.get('score', 'N/A')}")
 
 # Summary
 orig_score = original_eval.get('score', 0)
 impr_score = improved_eval.get('score', 0)
 print(f"\n=== Summary ===")
 print(f" Original score: {orig_score}")
 print(f" Improved score: {impr_score}")
 if impr_score > orig_score:
 print(" Result: Improved prompt WINS")
 elif impr_score == orig_score:
 print(" Result: TIE -- both prompts perform equally")
 else:
 print(" Result: Original prompt was better (optimization did not help)")
 
 return {
 "original_prompt": original_prompt,
 "improved_prompt": improved_prompt,
 "original_score": orig_score,
 "improved_score": impr_score
 }

# Test
test_cases_sol = [
 {"input": "The project was delivered on time and under budget.", "expected": "POSITIVE"},
 {"input": "We lost three key team members during the engagement.", "expected": "NEGATIVE"},
 {"input": "The analysis was thorough but the timeline was tight.", "expected": "NEUTRAL"},
 {"input": "Revenue increased 25% after implementing the recommendations.", "expected": "POSITIVE"},
 {"input": "The final presentation lacked the depth our board expected.", "expected": "NEGATIVE"},
]

results = auto_optimize_solution("Classify the sentiment.", test_cases_sol)